# Eksport BTC/USDT do CSV

Notebook służy do uruchomienia **lokalnie** (tam, gdzie Binance API jest dostępne).
Pobiera świece OHLCV przez `ccxt`, sprawdza ciągłość danych i zapisuje surowy zbiór do `btc_usdt_1h.csv`.

CSV jest później wejściem do notebooka treningowego MOMENT-1 w Google Colab.


In [1]:
import ccxt
import pandas as pd
from pathlib import Path

TIMEFRAME_MS = {
    "1m": 60 * 1000,
    "5m": 5 * 60 * 1000,
    "15m": 15 * 60 * 1000,
    "1h": 60 * 60 * 1000,
    "1d": 24 * 60 * 60 * 1000,
}

FREQ = {
    "1m": "min",
    "5m": "5min",
    "15m": "15min",
    "1h": "h",
    "1d": "D",
}

TICKER = "BTC/USDT"
TIMEFRAME = "1h"

START_ISO = "2023-11-03T20:00:00Z"
END_ISO = "2026-03-31T20:00:00Z"

OUTPUT_CSV = "btc_usdt_1h.csv"

CLIENT = ccxt.binance({
    "enableRateLimit": True,
})

START_DATE = CLIENT.parse8601(START_ISO)
END_DATE = CLIENT.parse8601(END_ISO)


In [ ]:
def fetch_ohlcv_range(client, ticker, timeframe, start_ms, end_ms):
    rows = []
    since = start_ms

    while since < end_ms:
        batch = client.fetch_ohlcv(
            ticker,
            timeframe=timeframe,
            since=since,
            limit=1000,
        )

        if not batch:
            break

        # Nie zapisujemy świec poza zakresem.
        batch = [row for row in batch if row[0] <= end_ms]

        if not batch:
            break

        rows.extend(batch)

        next_since = batch[-1][0] + TIMEFRAME_MS[timeframe]

        if next_since <= since:
            raise RuntimeError("API nie przesunęło zakresu czasu.")

        since = next_since

        print(
            "Pobrano:",
            len(rows),
            "| ostatnia świeca:",
            pd.to_datetime(batch[-1][0], unit="ms", utc=True),
        )

    return rows


In [3]:
rows = fetch_ohlcv_range(
    CLIENT,
    TICKER,
    TIMEFRAME,
    START_DATE,
    END_DATE,
)

data = pd.DataFrame(
    rows,
    columns=["Date", "Open", "High", "Low", "Close", "Volume"],
)

data["Date"] = pd.to_datetime(
    data["Date"],
    unit="ms",
    utc=True,
)

data = (
    data
    .drop_duplicates(subset="Date")
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Shape:", data.shape)
display(data.head())
display(data.tail())


Pobrano: 1000 | ostatnia świeca: 2023-12-15 11:00:00+00:00
Pobrano: 2000 | ostatnia świeca: 2024-01-26 03:00:00+00:00
Pobrano: 3000 | ostatnia świeca: 2024-03-07 19:00:00+00:00
Pobrano: 4000 | ostatnia świeca: 2024-04-18 11:00:00+00:00
Pobrano: 5000 | ostatnia świeca: 2024-05-30 03:00:00+00:00
Pobrano: 6000 | ostatnia świeca: 2024-07-10 19:00:00+00:00
Pobrano: 7000 | ostatnia świeca: 2024-08-21 11:00:00+00:00
Pobrano: 8000 | ostatnia świeca: 2024-10-02 03:00:00+00:00
Pobrano: 9000 | ostatnia świeca: 2024-11-12 19:00:00+00:00
Pobrano: 10000 | ostatnia świeca: 2024-12-24 11:00:00+00:00
Pobrano: 11000 | ostatnia świeca: 2025-02-04 03:00:00+00:00
Pobrano: 12000 | ostatnia świeca: 2025-03-17 19:00:00+00:00
Pobrano: 13000 | ostatnia świeca: 2025-04-28 11:00:00+00:00
Pobrano: 14000 | ostatnia świeca: 2025-06-09 03:00:00+00:00
Pobrano: 15000 | ostatnia świeca: 2025-07-20 19:00:00+00:00
Pobrano: 16000 | ostatnia świeca: 2025-08-31 11:00:00+00:00
Pobrano: 17000 | ostatnia świeca: 2025-10-12 03:0

,Date,Open,High,Low,Close,Volume
0,2023-11-03 20:00:00+00:00,34530.71,34659.66,34485.65,34593.90,1046.24789
1,2023-11-03 21:00:00+00:00,34593.90,34700.00,34583.86,34600.93,704.45031
2,2023-11-03 22:00:00+00:00,34600.94,34635.25,34563.54,34628.14,645.16337
3,2023-11-03 23:00:00+00:00,34628.13,34740.00,34611.38,34716.78,894.85032
4,2023-11-04 00:00:00+00:00,34716.78,34725.00,34653.84,34696.65,654.53035


,Date,Open,High,Low,Close,Volume
21092,2026-03-31 16:00:00+00:00,66736.79,68030.19,66729.28,67665.83,1334.03651
21093,2026-03-31 17:00:00+00:00,67665.83,68589.49,67403.81,67526.75,2195.81331
21094,2026-03-31 18:00:00+00:00,67526.76,67947.98,67526.30,67854.05,426.40012
21095,2026-03-31 19:00:00+00:00,67854.04,68029.41,67716.35,67838.80,435.24470
21096,2026-03-31 20:00:00+00:00,67838.79,68267.27,67831.70,68265.53,693.06575


In [4]:
# Kontrola ciągłości.
expected = pd.date_range(
    start=data["Date"].min(),
    end=data["Date"].max(),
    freq=FREQ[TIMEFRAME],
)

missing = expected.difference(
    pd.DatetimeIndex(data["Date"])
)

print("Pierwsza świeca:", data["Date"].min())
print("Ostatnia świeca:", data["Date"].max())
print("Liczba rekordów:", len(data))
print("Brakujące timestampy:", len(missing))

if len(missing):
    display(pd.Series(missing[:50], name="missing_timestamp"))


Pierwsza świeca: 2023-11-03 20:00:00+00:00
Ostatnia świeca: 2026-03-31 20:00:00+00:00
Liczba rekordów: 21097
Brakujące timestampy: 0


In [5]:
data.to_csv(
    OUTPUT_CSV,
    index=False,
)

print("Zapisano:", Path(OUTPUT_CSV).resolve())
print("Rozmiar [MB]:", round(Path(OUTPUT_CSV).stat().st_size / 1024**2, 2))


Zapisano: /Users/mikolaj/Desktop/STUDIA/CDV STOPIEŃ II/MAGISTERKA/master-thesis/notebooks/btc_usdt_1h.csv
Rozmiar [MB]: 1.45


### Google Colab

Po zapisaniu pliku przenieś `btc_usdt_1h.csv` do Colaba. Możesz go wgrać ręcznie do sesji albo umieścić na Google Drive. Notebook treningowy poniżej obsługuje oba warianty.
